**AI Business Research Copilot (RAG + Local Ollama)**

**Author:** Akshay Bhujbal

**Project Type:** Generative AI / NLP / Retrieval-Augmented Generation (RAG) / Local LLM Portfolio Project


**Project Overview**

This project demonstrates a **Business Research Copilot system** using:

* **LangChain + Ollama (Local LLM)**
* **ChromaDB (Vector database for semantic search)**
* **WebBaseLoader (Load website content dynamically)**

The project allows users to:

1. Load multiple trusted business/finance websites.
2. Split website content into chunks for efficient processing.
3. Create embeddings and store them in a vector database.
4. Query the LLM to get answers with source citations.
5. Build a fully offline, local AI-powered research assistant.

**Websites used (trusted sources):**

* [MoneyControl](https://www.moneycontrol.com)
* [Investing.com](https://www.investing.com)
* [Economic Times](https://www.economictimes.indiatimes.com)
* [Business Standard](https://www.business-standard.com)

**Tech Stack:**
  
-  Python  
-  LangChain  
-  ChromaDB (Vector DB)  
-  Ollama Llama3.2:1b (local model)  
-  Sentence Transformers (Embeddings)  
-  Streamlit (optional for deployment)


# Install Required Libraries
- ```langchain``` → **Orchestrates LLM + vector DB + retrieval.**
- ```chromadb``` → **Stores embeddings for retrieval.**
- ```sentence-transformers``` → **Converts text into embeddings.**
- ```beautifulsoup4``` + ```lxml``` + ```requests``` → **Scrape web pages.**

In [1]:
# Install all required packages
 !pip install langchain chromadb sentence-transformers beautifulsoup4 requests lxml

# Import Libraries

- ```WebBaseLoader``` → **Loads website text.**
- ```RecursiveCharacterTextSplitter``` → **Splits long text into smaller chunks for embedding.**
- ```SentenceTransformerEmbeddings``` → **Converts chunks to vectors.**
- ```Chroma``` → **Stores and retrieves vectors.**
- ```Ollama``` → **Runs Llama3.2:1b locally.**
- ```RetrievalQA``` → **RAG chain combining LLM + Retriever.**

In [2]:
from langchain.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.llms import Ollama
import os

USER_AGENT environment variable not set, consider setting it to identify your requests.


# Define Trusted URLs
These are **trusted sources** for financial and business info. You can add more or replace them with any website.

In [3]:
urls = [
    "https://www.moneycontrol.com/stocks/marketstats/",
    "https://www.economictimes.indiatimes.com/markets",
    "https://www.financialexpress.com/market/",
    "https://www.business-standard.com/"
]


# Load Website Data
- ```WebBaseLoader``` **scrapes text from websites.**
- ```docs``` **list stores all loaded documents.**
- **Errors are caught if a website fails to load.**

In [4]:
# Load all website content
loaders = [WebBaseLoader(url) for url in urls]
docs = []

for loader in loaders:
    try:
        data = loader.load()
        docs.extend(data)
    except Exception as e:
        print(f"Error loading {loader}: {e}")

print(f"Total documents loaded: {len(docs)}")

Total documents loaded: 4


# Split Text into Chunks
- **LLMs work better with smaller chunks of text.**
- **Overlap ensures context is not lost between chunks.**


In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
split_docs = splitter.split_documents(docs)

print(f"Total chunks created: {len(split_docs)}")

Total chunks created: 119


# Create Embeddings and Vector Database
- **Converts each text chunk into a numerical vector.**
- **Stores vectors in ChromaDB for fast retrieval.**

In [6]:
# Initialize embeddings
embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# Create vector database with Chroma
vector_store = Chroma.from_documents(split_docs, embedding_function)

print(" Vector store created successfully!")


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_9708\3043873310.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")



 Vector store created successfully!


# Initialize Ollama Local LLM
- **Uses Llama3.2:1b installed locally in Ollama.**
- **Fully offline; works with your RAG pipeline.**

In [7]:
# Initialize Ollama local LLM
llm = Ollama(model="llama3.2:1b")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_9708\461239926.py:2: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3.2:1b")


# Build RAG Chain
- **Retriever fetches top ```k``` relevant chunks.**
- **LLM generates an answer using retrieved context.**
- ```return_source_documents=True```→ **keeps source links for citations.**

In [8]:
# Create retriever from vector store
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# Combine LLM + Retriever into a RAG pipeline
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

# Ask a Business Question
- **This is the core RAG query.**
- **Response includes AI-generated answer + sources.**

In [10]:
# Example query
query = "What are the growth prospects and risks for renewable energy companies in India in 2025?"

# Run query through RAG
response = rag_chain.invoke({"query": query})

# Display answer
print(" Question:", query)
print("\n Answer:\n", response["result"])

# Display sources
print("\n Sources:")
for doc in response["source_documents"]:
    print("-", doc.metadata.get("source", "Unknown"))


 Question: What are the growth prospects and risks for renewable energy companies in India in 2025?

 Answer:
 Based on the provided context, it appears that there are mixed views about the growth prospects and risks for renewable energy companies in India in 2025. Here's a summary of the points mentioned:

Growth Prospects:

* According to the article, AI and trade reforms could power India's next growth wave.
* The World Bank's South Asia chief economist is also optimistic about India's potential to become a leader in global innovation.
* Sustaining above 25,000 may unlock fresh upside for the Nifty.

Risks:

* There are some risks mentioned in the article, such as RBI intensifying offshore market intervention to stabilize the rupee and planned provisioning norms unlikely to hit private banks hard.
* Additionally, there is a mention of FPIs increasing bearish bets on Nifty futures, suggesting that global investors may be skeptical about India's growth prospects.

However, it's also w

# Optional – Better Output Formatting
- **Makes answers readable for portfolio demos.**
- **Clearly separates summary from sources.**

In [11]:
def display_response(response, query):
    print("\033[1mQuestion:\033[0m", query, "\n")
    print("\033[1mAI Summary:\033[0m\n", response["result"], "\n")
    print("\033[1mSources:\033[0m")
    for doc in response["source_documents"]:
        print("-", doc.metadata.get("source", "Unknown"))

# Test improved display
query2 = "Which Indian IT companies are likely to benefit the most from AI adoption in 2025?"
response2 = rag_chain.invoke({"query": query2})
display_response(response2, query2)


Question: Which Indian IT companies are likely to benefit the most from AI adoption in 2025? 

AI Summary:
 Based on the provided context, it appears that Accenture is likely to benefit the most from AI adoption in 2025. The article mentions that Accenture's Q4 revenue rose 7% due to higher AI demand, and they are also described as "a leading Indian IT services company." This suggests that Accenture might be a key player in the Indian IT industry that could potentially benefit from AI adoption in 2025. 

Sources:
- https://www.economictimes.indiatimes.com/markets
- https://www.financialexpress.com/market/
- https://www.economictimes.indiatimes.com/markets


# Save Vector Database
- **Saves embeddings so you don’t have to rebuild every time.**
- **Makes the notebook production-ready.**

In [12]:
# Persist vector DB for future use
vector_store.persist()
print(" Knowledge base saved successfully!")

 Knowledge base saved successfully!


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_9708\2218592760.py:2: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_store.persist()


# App Screenshots Documentation

Below are the screenshots demonstrating the functionality and usage of the AI Business Research Copilot app.


**Overall App Look**  
![SC01_Overall_App](screenshots/SC01_Overall_App.PNG)

**URLs Section**  
![SC02_Added_URLs](Screenshots/SC02_Added_URLs.PNG)

**Process URLs Section**  
![SC03_Process_URLs](Screenshots/SC03_Process_URLs.png)

**Ask Question & Answer (Out of Context Question)**  
![SC04_Question_Answer](Screenshots/SC04_Question_Answer.png)

**Ask Question & Answer (Related to URLs / Contextual Question 1)**  
![SC05_Question_Answer](Screenshots/SC05_Question_Answer.png)

**Ask Question & Answer (Related to URLs / Contextual Question 2)**  
![SC06_Question_Answer](Screenshots/SC06_Question_Answer.png)


# Notes for Beginners

* **RAG (Retrieval-Augmented Generation):** LLM doesn’t rely solely on its training; it retrieves context from documents.
* **Chunking:** Important for large websites; prevents LLM from being overloaded.
* **Embeddings:** Converts text into numbers that capture semantic meaning.
* **Vector Store:** Lets the LLM find relevant text quickly for accurate answers.
* **Ollama Local LLM:** Fully offline, secure, and fast response for sensitive data.

# Conclusion & Summary

The **AI Business Research Copilot** is a versatile tool designed to assist users in gathering insights from multiple websites and answering questions based on the processed information.  

## Key Highlights:

-  Users can **add multiple URLs** dynamically and process them for content extraction.
-  The app **hides URLs** for a cleaner interface and allows reviewing them via a dropdown.
-  Supports **contextual question answering** using a local LLM, providing intelligent responses based on the processed data.
-  Designed for **ease of use**, making research faster and more efficient.

## Summary:
    
This project demonstrates how AI can automate research tasks, extract relevant information from multiple sources, and provide meaningful answers to user queries. It can serve as a foundation for more advanced business research tools, integrating further features like trend analysis, summarization, or interactive dashboards.

> Overall, this app combines **data processing, LLM capabilities, and a user-friendly interface** into a professional research assistant.
